In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from pymongo import MongoClient
import time


client = MongoClient("mongodb://localhost:27017")
db = client["jeju_yeogi"]
collection = db["yeogi"]


service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument("--window-size=1920x1080")
options.add_argument("--start-maximized")
options.add_argument("--lang=ko_KR")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)


base_url = "https://www.yeogi.com/domestic-accommodations?keyword=제주&checkIn=2025-09-12&checkOut=2025-09-13&personal=2&freeForm=false&page=1"
driver.get(base_url)


for page_num in range(1, 4):
    print(f"\n👉 {page_num} 페이지 ")
    

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)


    hotel_cards = driver.find_elements(By.CSS_SELECTOR, "a.gc-thumbnail-type-seller-card")
    
    page_data = []
    for card in hotel_cards:
        try:
            url = card.get_attribute("href")
            name = card.find_element(By.CSS_SELECTOR, "div.css-1j7tt62 > div > h3").text.strip()
            
            location = card.find_element(By.CSS_SELECTOR, "span.css-ki0lqh").text.strip()
            
            price_text = card.find_element(By.CSS_SELECTOR, "span.css-5r5920").text.strip()
            price_min = int(price_text.replace(",", "").replace("원", "")) if price_text else None
            
            rating_text = card.find_element(By.CSS_SELECTOR, "span.css-9ml4lz").text.strip()
            rating = float(rating_text) if rating_text else None
            
            review_text = card.find_element(By.CSS_SELECTOR, "span.css-oj6onp").text.strip()
            review_count = int(review_text.replace("명 평가", "").replace(",", "")) if review_text else 0
            
            record = {
                "name": name,
                "location": location,
                "price_min": price_min,
                "rating": rating,
                "review_count": review_count,
                "url": url,
                "page": page_num
            }
            page_data.append(record)
            print(record)
        except Exception as e:
            print("데이터 추출 실패:", e)
    
    if page_data:
        collection.insert_many(page_data)
        print(f"👉 {len(page_data)}개 데이터 MongoDB 저장 완료!")


    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='다음']")))
        next_btn.click()
        time.sleep(2)
    except Exception as e:
        print("다음 버튼 클릭 실패:", e)
        break

driver.quit()
client.close()


👉 1 페이지 
데이터 추출 실패: Message: no such element: Unable to locate element: {"method":"css selector","selector":"span.css-5r5920"}
  (Session info: chrome=140.0.7339.81); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0x9ad2a3+66419]
	GetHandleVerifier [0x0x9ad2e4+66484]
	(No symbol) [0x0x784bd3]
	(No symbol) [0x0x7ce958]
	(No symbol) [0x0x7cecfb]
	(No symbol) [0x0x7c3f21]
	(No symbol) [0x0x7f1064]
	(No symbol) [0x0x7c3e24]
	(No symbol) [0x0x7f11d4]
	(No symbol) [0x0x8128a1]
	(No symbol) [0x0x7f0e16]
	(No symbol) [0x0x7c25ce]
	(No symbol) [0x0x7c34a4]
	GetHandleVerifier [0x0xbf5ee3+2461619]
	GetHandleVerifier [0x0xbf0f66+2441270]
	GetHandleVerifier [0x0x9d6242+234258]
	GetHandleVerifier [0x0x9c6208+168664]
	GetHandleVerifier [0x0x9cd1ad+197245]
	GetHandleVerifier [0x0x9b55f8+100040]
	GetHandleVerifier [0x0x9b5792+100450]
	GetHandleVerifier [0x0x99f74a+10266]
	Ba

In [12]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["jeju_yeogi"]
collection = db["yeogi"]

pipeline = [
    {"$sort": {"rating": -1,"review_count": -1}},
    {"$limit": 10},
    {"$project": {
            "_id": 0,
            "name": 1,
            "rating": 1,
            "review_count": 1,
            "price_min": 1,
            "location": 1,
            "url": 1
        }
    }
]

results = list(collection.aggregate(pipeline))

print("\n👉 평점 상위 10 숙소")
for r in results:
    print(f"{r['name']} | 평점: {r['rating']} | 리뷰수: {r['review_count']} | 가격: {r['price_min']} | 위치: {r['location']} | URL: {r['url']}")

client.close()


👉 평점 상위 10 숙소
그랜드 조선 제주 | 평점: 9.7 | 리뷰수: 751 | 가격: 471040 | 위치: 중문관광단지 내 | URL: https://www.yeogi.com/domestic-accommodations/66687?checkIn=2025-09-12&checkOut=2025-09-13&personal=2
제주 아름드리 펜션 | 평점: 9.7 | 리뷰수: 559 | 가격: 88000 | 위치: 월정리해수욕장 도보 3분 | URL: https://www.yeogi.com/domestic-accommodations/56703?checkIn=2025-09-12&checkOut=2025-09-13&personal=2
롯데시티호텔 제주 | 평점: 9.5 | 리뷰수: 992 | 가격: 226000 | 위치: 제주공항 차량5분 | URL: https://www.yeogi.com/domestic-accommodations/6820?checkIn=2025-09-12&checkOut=2025-09-13&personal=2
제주썬 호텔 | 평점: 9.5 | 리뷰수: 403 | 가격: 204000 | 위치: 제주국제공항 차량 8분 | URL: https://www.yeogi.com/domestic-accommodations/71915?checkIn=2025-09-12&checkOut=2025-09-13&personal=2
랜딩관 제주신화월드 호텔앤리조트 | 평점: 9.4 | 리뷰수: 2688 | 가격: 252400 | 위치: 신화테마파크 & 신화워터파크 도보 3분 | URL: https://www.yeogi.com/domestic-accommodations/48099?checkIn=2025-09-12&checkOut=2025-09-13&personal=2
신화관 제주신화월드 호텔앤리조트 | 평점: 9.4 | 리뷰수: 1923 | 가격: 389000 | 위치: 신화테마파크 & 신화워터파크 도보 3분 | URL: https://www.yeogi.com/domesti

In [14]:
from pymongo import MongoClient


client = MongoClient("mongodb://localhost:27017")
db = client.jeju_yeogi
hotels = db.yeogi

top_hotels_pipeline = [
    {"$sort": {"rating": -1, "review_count": -1}},
    {"$limit": 10},
    {"$project": {
        "_id": 0,
        "name": "$name",
        "rating": "$rating",
        "review_count": "$review_count",
        "price_min": "$price_min",
        "location": "$location",
        "url": "$url"
    }}
]

top_hotels = list(hotels.aggregate(top_hotels_pipeline))

print("\n🏨 제주도 평점 상위 10 숙소 리스트 \n")
for idx, hotel in enumerate(top_hotels, start=1):
    print(f" 숙소 #{idx}: {hotel['name']}")
    print(f"  평점      : {hotel['rating']}")
    print(f"  리뷰 수    : {hotel['review_count']}")
    print(f"  최저가격  : {hotel['price_min']}원")
    print(f"  위치      : {hotel['location']}")
    print(f"  예약 링크 : {hotel['url']}\n")

client.close()


🏨 제주도 평점 상위 10 숙소 리스트 

 숙소 #1: 그랜드 조선 제주
  평점      : 9.7
  리뷰 수    : 751
  최저가격  : 471040원
  위치      : 중문관광단지 내
  예약 링크 : https://www.yeogi.com/domestic-accommodations/66687?checkIn=2025-09-12&checkOut=2025-09-13&personal=2

 숙소 #2: 제주 아름드리 펜션
  평점      : 9.7
  리뷰 수    : 559
  최저가격  : 88000원
  위치      : 월정리해수욕장 도보 3분
  예약 링크 : https://www.yeogi.com/domestic-accommodations/56703?checkIn=2025-09-12&checkOut=2025-09-13&personal=2

 숙소 #3: 롯데시티호텔 제주
  평점      : 9.5
  리뷰 수    : 992
  최저가격  : 226000원
  위치      : 제주공항 차량5분
  예약 링크 : https://www.yeogi.com/domestic-accommodations/6820?checkIn=2025-09-12&checkOut=2025-09-13&personal=2

 숙소 #4: 제주썬 호텔
  평점      : 9.5
  리뷰 수    : 403
  최저가격  : 204000원
  위치      : 제주국제공항 차량 8분
  예약 링크 : https://www.yeogi.com/domestic-accommodations/71915?checkIn=2025-09-12&checkOut=2025-09-13&personal=2

 숙소 #5: 랜딩관 제주신화월드 호텔앤리조트
  평점      : 9.4
  리뷰 수    : 2688
  최저가격  : 252400원
  위치      : 신화테마파크 & 신화워터파크 도보 3분
  예약 링크 : https://www.yeogi.com/domestic-accommoda